## 10.1 MLP 案例 - 数据准备

#### 1. 为什么选择 MNIST 数据集

##### 1.1 什么是 MNIST
MNIST 是一个非常经典的手写数字识别数据集，数据中的图片内容是 0 到 9 的手写数字。

模型的任务就是：输入一张手写数字图片，输出它属于哪一个数字类别。

也就是说，这是一个典型的：

👉 多分类问题（Multi-class Classification）

因为输出结果不是“是/否”两类，而是：

`0-9`

一共10个class

##### 1.2 为什么它特别适合 MLP
虽然图片任务后面更适合用 CNN，但在学习 MLP 时，MNIST 非常适合作为综合案例，因为：

（1）数据规模合适

MNIST 不算太大，训练起来压力不高，适合练习完整流程。

（2）图片尺寸统一

每张图片都是 28 × 28 像素，处理起来很方便。

（3）可以很自然地转成 MLP 输入

MLP 的输入通常要求是一维向量。

而一张 28 × 28 的图片，本来是一个二维矩阵：

`28 × 28`

我们只需要把它“拉平（flatten）”成一维向量：

`28 * 28 = 784`

于是，每张图片就可以作为一个长度为 784 的特征向量输入到 MLP 中。

（4）标签清晰，便于理解损失函数

因为是 10 分类任务，所以非常适合配合我们之前学过的：
* 激活函数 - softmax
* 损失函数 - crossEntropyLoss

#### 2. 这个案例到底在做什么

##### 2.1 输入是什么
输入是一张手写数字图片，例如：
* 一张写着 “3” 的图片
* 一张写着 “7” 的图片

每张图片大小固定为：

`28 × 28`

##### 2.2 输出是什么
输出是模型对这张图片所属类别的预测概率（softmax）：
* 预测它是 0的概率
* 预测它是 1的概率
* …
* 预测它是 9的概率

所以输出层应该有：

`10 个神经元`

每一个神经元对应一个类别的概率。

##### 2.3 整个任务本质上是什么
本质上就是：

把一张图片中的像素信息，转换成一个数字类别。

更具体一点说：
* 图片中的每个像素值都是特征
* MLP 学习这些像素和数字类别之间的关系
* 最后输出最可能的数字类别

#### 3. MNIST 数据集的基本结构

##### 3.1 数据集包含什么
MNIST 数据集通常分为两部分：
* 训练集（training set）
* 测试集（test set）

经典规模是：
* 训练集：60,000 张图片
* 测试集：10,000 张图片

##### 3.2 每条数据由什么组成
每条样本通常包含两部分：

（1）特征 X

一张手写数字图片，28*28

（2）标签 y

这张图片对应的真实数字类别

##### 3.3 标签的形状
标签一般是一个整数，还没有进行 one-hot。

例如一个 batch 的标签可能是：

`[5, 0, 4, 1, 9, ...]`

形状通常为：

`[batch_size]`

例如：

`[64]`

##### 3.4 注意：进行 One-Hot 编码
为了符合PyTorch 中 CrossEntropyLoss 的要求

但是在神经网络公式计算中，单个标签无法与输出层的多个神经元结果进行计算

所以我们需要把标签表示为一个 向量：

`[0,0,0,1,0,0,0,0,0,0,0]`

因此引入了：

`One-Hot 编码`

#### 4. 为什么图片可以作为 MLP 的输入

##### 4.1 MLP 的输入本质
MLP 并不理解“图片”这个概念。

它只接收：

👉 一组数值特征

所以对于 MLP 来说，一张图片其实就是很多像素值组成的数字列表。


##### 4.2 图片如何变成特征向量
原始图片：

`28 × 28`

本质上是 28 行 28 列的像素矩阵。
```
[
  [0, 0, 12, ..., 0],
  [0, 34, 200, ..., 0],
  ...
]
```

把它按顺序拉平后，就变成：

`[0, 0, 12, ..., 34, 200, ..., 0]`

长度就是：

`784`

于是对于 MLP 而言，一张图片就是：

👉 一个拥有 784 个输入特征 的样本

##### 4.3 这里的每个特征代表什么
每一个输入特征，其实就是图片中某一个像素点的灰度值。

所以 MLP 的工作可以理解为：

根据 784 个像素值，判断这张图片更像 0~9 中的哪一个数字。

#### 5. MNIST 在 PyTorch 中的基本加载方式

In [1]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 1. 创建数据预处理
transform = transforms.ToTensor() # 将图像转换为张量，并将像素值归一化到[0, 1]范围

# 2. 加载MNIST数据集
train_dataset = datasets.MNIST(
    root='./data', # 数据集存储路径
    train=True, # 训练集
    download=True, # 如果数据集不存在，则下载数据集
    transform=transform # 应用数据预处理
)
test_dataset = datasets.MNIST(
    root='./data', # 数据集存储路径
    train=False, # 测试集
    download=True, # 如果数据集不存在，则下载数据集
    transform=transform # 应用数据预处理
)

# 3. 创建 dataloader
train_loader = DataLoader(
    dataset=train_dataset, # 训练数据集
    batch_size=64, # 每个批次的样本数量
    shuffle=True # 是否打乱数据
)
test_loader = DataLoader(
    dataset=test_dataset, # 测试数据集
    batch_size=64, # 每个批次的样本数量
    shuffle=False # 是否打乱数据
)

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9.91M/9.91M [00:00<00:00, 13.2MB/s]


Extracting ./data/MNIST/raw/train-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28.9k/28.9k [00:00<00:00, 515kB/s]


Extracting ./data/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1.65M/1.65M [00:00<00:00, 4.81MB/s]


Extracting ./data/MNIST/raw/t10k-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4.54k/4.54k [00:00<00:00, 1.55MB/s]

Extracting ./data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/MNIST/raw



##### 5.1 transforms.ToTensor()
`transform = transforms.ToTensor()`

作用：
* 把图片转成 PyTorch Tensor
* 把像素值从 0~255 缩放到 0~1

这一步非常常见，是图像任务的基础预处理。

##### 5.2 datasets.MNIST(...)
`train_dataset = datasets.MNIST(...)`

它的作用是：
* 自动下载 MNIST 数据集
* 自动读取数据
* 返回一个可被 PyTorch 使用的数据集对象

常见参数解释：
* root="./data"
    * 表示数据集下载和保存的位置。
* train=True
    * 表示加载训练集。
* train=False
    * 表示加载测试集。
* download=True
    * 如果本地没有数据，就自动下载。
* transform=transform
    * 表示对每张图片应用我们定义好的预处理。


##### 5.3 DataLoader(...)
DataLoader 的作用是：
* 按批次读取数据
* 可选择是否打乱数据
* 方便训练时循环迭代

例如：

`train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)`

表示：
* 每次取 64 张图片
* 每个 epoch 都打乱训练数据顺序

而测试集一般不需要打乱，所以：

`shuffle=False`

#### 6. 如何查看一条样本的内容

In [2]:
image, label = train_dataset[0] # 获取训练数据集中的第一个样本
print(image.shape) # 输出图像的形状
print(label) # 输出图像的标签

torch.Size([1, 28, 28])
5


##### 6.1 image.shape = [1, 28, 28] 是什么意思
这里的 1 表示：

`通道数 channel = 1`

因为 MNIST 是灰度图，不是彩色图。

所以一张图片的张量形状是：

`[1, 28, 28]`

含义是：
* 1 个通道，非RGB
* 高 28
* 宽 28

##### 6.2 label = 5 是什么意思
表示这张图片真实对应的数字类别是 5。

注意这里的标签还没有经过 one-hot，而是一个整数类别编号。

在后面使用损失函数时，会自动转换为 one-hot 

`nn.CrossEntropyLoss()`

##### 7. 如何查看一个 batch 的形状

In [3]:
images, labels = next(iter(train_loader)) # 获取训练数据加载器中的第一个批次
print(images.shape) # 输出批次图像的形状
print(labels.shape) # 输出批次标签的形状

torch.Size([64, 1, 28, 28])
torch.Size([64])


##### 7.1 images.shape = [64, 1, 28, 28]
含义是：
* 64 张图片
* 每张图片 1 个通道
* 每张图片大小 28 × 28

##### 7.2 labels.shape = [64]
表示这 64 张图片分别对应 64 个类别标签

#### 8. 在进入 MLP 前如何展平数据

##### 8.1 使用 view() 或 reshape()

In [5]:
images = images.reshape(-1, 28*28) # -1 表示自动推断维度，这里是batch_size，28*28表示每个图像的像素数量
print(images.shape) # 输出批次图像的形状

torch.Size([64, 784])
